<center>
    <p align="center">
        <img src="https://logodownload.org/wp-content/uploads/2017/09/mackenzie-logo-3.png" style="height: 7ch;"><br>
        <h1 align="center">Computer Systems Undergraduate Thesis</h1>
        <h2 align="center">Quantitative Analysis of the Impact of Image Pre-Processing on the Accuracy of Computer Vision Models Trained to Identify Dermatological Skin Diseases</h2>
        <h4 align="center">Gabriel Mitelman Tkacz</h4>
    </p>
</center>

<hr>

In [ ]:
import json
from pathlib import Path

import matplotlib.pyplot as plt
import pandas as pd
from IPython.display import display
from PIL import Image

OUTPUT_DIR = Path("../output")

with (OUTPUT_DIR / "results_matrix.json").open() as f:
    results = json.load(f)

df = pd.DataFrame(results)
print(f"Loaded {len(df)} results")
df.head()

## Results Summary

In [ ]:
summary_cols = ["combo_key", "confidence_level", "accuracy", "alpha", "gamma", "weighted_alpha"]
summary = df[summary_cols].sort_values("weighted_alpha", ascending=False)
display(summary.head(20))

## Accuracy by Confidence Level

In [ ]:
for level in df["confidence_level"].unique():
    subset = df[df["confidence_level"] == level].sort_values("accuracy", ascending=False)
    print(f"\n=== {level} ===")
    display(subset[["combo_key", "accuracy", "alpha", "gamma", "weighted_alpha"]].head(10))

## Confusion Matrices

In [ ]:
cm_df = df[["combo_key", "confidence_level", "confusion_matrix"]].copy()
cm_expanded = pd.json_normalize(cm_df["confusion_matrix"])
cm_df = pd.concat([cm_df.drop(columns=["confusion_matrix"]).reset_index(drop=True), cm_expanded], axis=1)
display(cm_df.head(20))

## Grad-CAM Overlays

In [ ]:
for _, row in df.iterrows():
    gradcam_images = row.get("gradcam_images", [])
    if not gradcam_images:
        continue

    combo_dir = OUTPUT_DIR / row["confidence_level"] / row["combo_key"]
    n_images = len(gradcam_images)

    fig, axes = plt.subplots(n_images, 2, figsize=(8, 4 * n_images))
    fig.suptitle(f"{row['combo_key']} ({row['confidence_level']})", fontsize=14)

    if n_images == 1:
        axes = [axes]

    for i, gc in enumerate(gradcam_images):
        pos_path = combo_dir / gc["pos_overlay"]
        neg_path = combo_dir / gc["neg_overlay"]

        if pos_path.exists():
            axes[i][0].imshow(Image.open(pos_path))
            axes[i][0].set_title(f"Positive - {gc['image_id']} (conf: {gc['confidence']:.3f})")
        axes[i][0].axis("off")

        if neg_path.exists():
            axes[i][1].imshow(Image.open(neg_path))
            axes[i][1].set_title(f"Negative - {gc['image_id']}")
        axes[i][1].axis("off")

    plt.tight_layout()
    plt.show()